# Native RAG

<img src="../images/Native RAG.png" width=“500”>

In [1]:
from langchain_core.globals import set_debug
set_debug(False)

## Initialize Azure Chat OpenAI

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv(override=True, verbose=True)

chat_llm = AzureChatOpenAI(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')
)

## Initialize Azure Embedding Open AI

In [3]:
emd_llm = AzureOpenAIEmbeddings(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
)

## Load the Document and upload it to vector store

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

file = '../Data/HTTP-Status-Codes.pdf'
pdf_docs = PyMuPDFLoader(file_path=file).load()


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents=pdf_docs)

In [6]:
from langchain_community.vectorstores import FAISS

faiss = FAISS.from_documents(documents=chunks, embedding=emd_llm)


## User question 

In [16]:
num_queries = 4
user_question = "What is error code 404?"

## Dense retrieval

In [17]:

faiss_chunks = faiss.similarity_search_with_score(query=user_question, k=num_queries)

In [18]:
print(f"Chunks fetched from FAISS Vector store: {len(faiss_chunks)}")
faiss_top_chunks = []
for c, score in faiss_chunks:
    print(score)
    print(c.page_content, end="\n-----\n")
    
    faiss_top_chunks.append(c.page_content)

Chunks fetched from FAISS Vector store: 4
0.9218305
permanent.
4XX codes flag client errors. You typed a wrong URL. You don’t have
permission. The page doesn’t exist. These are on you or your users.
-----
0.9622687
404 Not Found Error Impact
410 Gone vs 404 Error Codes
500 Internal Server Error Solutions
503 Service Unavailable Fix
Common HTTP Status Code FAQ
Search...
Tutorials
Comparisons
News
Keep in touch
-----
0.9688201
client side.
400 Bad Request
The server can’t process your request. Client-side error. Invalid syntax. Bad
routing. Wrong parameters. Check your URL and clear browser cache to fix it.
-----
0.9837285
302 Found
303 See Other
304 Not Modified
307 Temporary Redirect
308 Permanent Redirect
4XX Client Error Codes Reference
400 Bad Request
401 Unauthorized
402 Payment Required
403 Forbidden
-----


In [19]:
from langchain_core.prompts import ChatPromptTemplate
PROMPT_TEMPLATE = """"
You are a helpful assistance. You answer user's query {question} from the provided context {context}. If context isn't sufficient to provide the answer, inform the user that Context isn't sufficient to provide the answer of your query.
"""
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", PROMPT_TEMPLATE),
        ("user",  "user query: {question}"),
    ]
)


In [20]:
prompt =  prompt_template.invoke({'question': user_question, 'context': user_question})
response = chat_llm.invoke(prompt)
print(response)

print(response.content)

content="Context isn't sufficient to provide the answer of your query." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 277, 'prompt_tokens': 73, 'total_tokens': 350, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 26, 'engine_ttlt_ms': 2148, 'pre_inference_ms': 129, 'service_tbt_ms': 8, 'service_ttft_ms': 548, 'service_ttlt_ms': 2654, 'total_duration_ms': 2535, 'user_visible_ttft_ms': 419}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DxADMicRCXKAcMnbBHjtYgWf0dQky', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': Fal